# Phase 40 next-6e — multi-seed byzantine + descent sweep (Kaggle)

Same purpose as the Colab notebook, ported to Kaggle's `/kaggle/working/` layout.

## Before clicking ▶ Run All

Open the **right sidebar** ("Notebook options" / gear icon) and confirm:

1. **Accelerator** → `GPU T4 x2` (or any GPU). T4 free quota is ~30 hr/week on Kaggle.
2. **Internet** → `On` (required for `pip install`, `git clone`, and to reach the deployed Worker).
3. **Persistence** → off is fine, the sweep finishes in one go.

Each run hits `https://postnet-cf.abgunaydin94.workers.dev`. 5 seeds × R=100 = ~20–25 min wall clock on T4. Final markdown table is at `/kaggle/working/sweep/TABLE2.md`.

## Cell 1 — Install deps + clone repos

In [ ]:
import subprocess, os, sys

# ── preflight: internet must be on ────────────────────────────────────────
import urllib.request
try:
    urllib.request.urlopen("https://github.com", timeout=5).close()
    print("✓ internet reachable")
except Exception as e:
    sys.exit(f"✗ INTERNET DISABLED: {e}\n"
             "  Right sidebar (gear icon) → Internet → On → verify phone if asked, then re-run this cell.")

# ── install deps ──────────────────────────────────────────────────────────
print("→ pip install …")
subprocess.run(["pip", "-q", "install", "transformers", "torch", "numpy", "requests"], check=True)

# ── clone repos (fresh, non-silent) ───────────────────────────────────────
os.makedirs("/kaggle/working", exist_ok=True)
for name, url in [("ntkmirror", "https://github.com/leochlon/ntkmirror.git"),
                  ("postnet-cf", "https://github.com/abgnydn/postnet-cf.git")]:
    dst = f"/kaggle/working/{name}"
    if os.path.isdir(dst):
        print(f"→ git pull {name}")
        subprocess.run(["git", "-C", dst, "pull", "--ff-only"], check=True)
    else:
        print(f"→ git clone {name}")
        subprocess.run(["git", "clone", url, dst], check=True)

# ── pip install ntkmirror (editable) ──────────────────────────────────────
print("→ pip install -e ntkmirror")
subprocess.run(["pip", "-q", "install", "-e", "/kaggle/working/ntkmirror"], check=True)
print("OK")

## Cell 2 — GPU check + coordinator sanity probe

In [ ]:
import torch, requests, json
assert torch.cuda.is_available(), "GPU not attached. Right sidebar → Accelerator → GPU T4 x2."
print("✓ GPU:", torch.cuda.get_device_name(0))

COORD = "https://postnet-cf.abgunaydin94.workers.dev"
_UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
       "AppleWebKit/537.36 (KHTML, like Gecko) "
       "Chrome/126.0.0.0 Safari/537.36 postnet-ntk/1.0")
_S = requests.Session(); _S.headers.update({'User-Agent': _UA})

r = _S.post(f"{COORD}/api/ntk/reset"); print('reset:', r.json())
s = _S.get(f"{COORD}/api/ntk/state").json()
assert 'pending_audits' in s, "deployed Worker missing next-6 fields — re-deploy main first."
print(f"R={s['round']}  eta={s['eta']}  pending_audits={s['pending_audits']}  K={s.get('K')}")

## Cell 3 — Python attacker (mirrors browser `?attack=1` mode)

Fabricates `(seed, scalar_g=1.0, claimed_delta=-10)`. **Does NOT send `audit_loss_before`** (per next-6 fix). UA-spoofed for Cloudflare.

In [ ]:
import threading, time, random

_HEADERS = {"User-Agent": _UA, "Content-Type": "application/json"}

class Attacker(threading.Thread):
    def __init__(self, coord, worker_id=None, poll_delay=0.25):
        super().__init__(daemon=True)
        self.coord = coord
        self.worker_id = worker_id or f"py-attacker-{random.randint(1,99999):05d}"
        self.poll_delay = poll_delay
        self.running = False
        self.local_round = 0
        self.rounds_advanced = 0
        self.quarantine_seen = False
        self.last_http_status = None

    def stop(self): self.running = False

    def run(self):
        self.running = True
        s = requests.Session(); s.headers.update(_HEADERS)
        try:
            p = s.post(f"{self.coord}/api/ntk/tick", json={"worker_id": self.worker_id}, timeout=10)
            self.last_http_status = p.status_code
            self.local_round = int(p.json().get('round', 0))
        except Exception: pass
        while self.running:
            try:
                fake_seed = random.randint(0, 2**32 - 1)
                body = {"worker_id": self.worker_id, "round": self.local_round,
                        "seed": fake_seed, "scalar_g": 1.0, "delta": -10.0,
                        "since_round": self.local_round}
                r = s.post(f"{self.coord}/api/ntk/tick", json=body, timeout=10)
                self.last_http_status = r.status_code
                j = r.json()
                if j.get('quarantined'): self.quarantine_seen = True
                if j.get('advanced'): self.rounds_advanced += 1
                if isinstance(j.get('round'), int): self.local_round = j['round']
            except Exception: pass
            time.sleep(self.poll_delay)

print('Attacker class defined.')

## Cell 4 — Honest verifier subprocess wrapper

In [ ]:
import subprocess, os

def run_honest(seed, rounds=100, log_path=None):
    cmd = [
        "python", "/kaggle/working/postnet-cf/scripts/ntk-verifier.py",
        "--coord", COORD,
        "--model", "Qwen/Qwen2.5-0.5B-Instruct",
        "--train", "/kaggle/working/ntkmirror/examples/math_train.jsonl",
        "--artifact", "/kaggle/working/postnet-cf/public/data/qwen05b-math-gates-k5000.bin",
        "--rounds", str(rounds),
        "--trials", "4",
        "--device", "cuda",
        "--dtype", "fp32",
        "--seed", str(seed),
        "--worker-id", f"kaggle-honest-s{seed}",
    ]
    if log_path:
        with open(log_path, 'w') as f:
            return subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
    return subprocess.run(cmd, capture_output=True, text=True)

print('honest runner ready (CUDA)')

## Cell 5 — The sweep

In [ ]:
SEEDS = [1, 2, 3, 4, 5]
ROUNDS = 100
os.makedirs('/kaggle/working/sweep', exist_ok=True)

results = []
for S in SEEDS:
    print(f"\n=== seed={S} ===")
    _S.post(f"{COORD}/api/ntk/reset")
    time.sleep(1)
    att = Attacker(COORD); att.start()
    t0 = time.time()
    rc = run_honest(S, rounds=ROUNDS, log_path=f"/kaggle/working/sweep/seed-{S}.log")
    elapsed = time.time() - t0
    att.stop(); att.join(timeout=5)
    state = _S.get(f"{COORD}/api/ntk/state").json()
    ws = state.get('worker_stats') or {}
    attacker_stats = next((s for w,s in ws.items() if w.startswith('py-attacker')), None)
    rec = {
        'seed': S, 'elapsed_s': round(elapsed, 1),
        'server_R': state.get('round'),
        'last_loss': state.get('last_loss'),
        'eta': state.get('eta'),
        'grow': state.get('eta_grow_events'),
        'shrink': state.get('eta_shrink_events'),
        'accept_rate': state.get('accept_rate'),
        'pending_audits': state.get('pending_audits'),
        'attacker_wins': attacker_stats['wins'] if attacker_stats else 0,
        'attacker_frauds': attacker_stats['frauds'] if attacker_stats else 0,
        'attacker_fraud_rate': attacker_stats['fraud_rate'] if attacker_stats else 0,
        'attacker_quarantine_seen': att.quarantine_seen,
        'attacker_last_http': att.last_http_status,
    }
    print(json.dumps(rec, indent=2))
    with open(f'/kaggle/working/sweep/seed-{S}.json', 'w') as f:
        json.dump({**rec, 'state_at_end': state}, f, indent=2)
    results.append(rec)

print('\n=== sweep complete ===')
print(json.dumps(results, indent=2))

## Cell 6 — Aggregate → markdown table

In [ ]:
import statistics as stats

def mean_std(xs):
    xs = [x for x in xs if x is not None]
    if not xs: return None, None
    return (stats.mean(xs), stats.stdev(xs) if len(xs) > 1 else 0.0)

def fmt(v, spec): return format(v, spec) if v is not None else '—'
def mfmt(m, s, spec): return f"{fmt(m, spec)} ± {fmt(s, spec)}"

lines = [
    '| seed | server R | last_loss | η | grow | accept | atk W/F | quarantined |',
    '|---|---|---|---|---|---|---|---|',
]
for r in results:
    lines.append(
        f"| {r['seed']} | {r['server_R']} | {fmt(r['last_loss'], '.6f')} | "
        f"{fmt(r['eta'], '.5f')} | {r['grow']} | {fmt(r['accept_rate'], '.3f')} | "
        f"{r['attacker_wins']}/{r['attacker_frauds']} | {'yes' if r['attacker_quarantine_seen'] else 'no'} |"
    )

loss_m, loss_s = mean_std([r['last_loss'] for r in results])
eta_m,  eta_s  = mean_std([r['eta'] for r in results])
grow_m, grow_s = mean_std([r['grow'] for r in results])
rate_m, rate_s = mean_std([r['attacker_fraud_rate'] for r in results])

lines.append(
    f"| **mean ± σ** | — | **{mfmt(loss_m, loss_s, '.6f')}** | "
    f"**{mfmt(eta_m, eta_s, '.5f')}** | **{mfmt(grow_m, grow_s, '.1f')}** | — | — | "
    f"**rate={mfmt(rate_m, rate_s, '.3f')}** |"
)

table = '\n'.join(lines); print(table)

bad = [r['seed'] for r in results if r['last_loss'] is None]
if bad:
    print(f"\n⚠ seeds with no honest audit: {bad}")
    print('  → check /kaggle/working/sweep/seed-<N>.log for verifier traceback')

with open('/kaggle/working/sweep/TABLE2.md', 'w') as f:
    f.write(table)
print('\nWrote /kaggle/working/sweep/TABLE2.md — paste this back to the chat')

## Cell 7 — Outputs (Kaggle auto-saves /kaggle/working)

Everything in `/kaggle/working/sweep/` is downloadable from the right sidebar **Output** panel after this notebook commits/saves. You don't need a special download cell.

In [ ]:
!ls -la /kaggle/working/sweep/